# EasyMagpieTTS — HTTP and incremental WebSocket requests

This notebook demonstrates both serving APIs exposed by the same `vllm serve` process:

1. `POST /v1/audio/speech` for a complete text input.
2. `WS /v1/audio/speech/stream` for incremental EasyMagpie token-ID chunks and asynchronous audio output.

Prepare the converted model and environment by following the sidecar README's
[setup instructions](../../../tools/easymagpie_vllm_omni/README.md#setup-the-serving-environment).
Then [start the server](../../../tools/easymagpie_vllm_omni/README.md#serve-over-http-and-websocket), launch this
notebook from the Speech repository using the same environment's Jupyter kernel, and set `MODEL_DIR` below to
the model served at `localhost:8091`.

In [ ]:
import asyncio
import json
import time
from pathlib import Path

import numpy as np
import requests
import websockets
from IPython.display import Audio, display
from transformers import AutoTokenizer

SERVER_URL = "http://localhost:8091"
SPEECH_ENDPOINT = f"{SERVER_URL}/v1/audio/speech"
STREAM_ENDPOINT = "ws://localhost:8091/v1/audio/speech/stream"
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "easymagpie_vllm_omni").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within a Speech repository checkout")


REPO_ROOT = find_repo_root()
MODEL_DIR = REPO_ROOT / "tools" / "easymagpie_vllm_omni" / "converted_model"
SAMPLE_RATE = 22050

print("health:", requests.get(f"{SERVER_URL}/health", timeout=5).status_code)
print("model:", MODEL_DIR.resolve())

In [ ]:
def synthesize_http(
    text: str,
    speaker_id: str = "eng",
    max_new_tokens: int = 1024,
    timeout: float = 300.0,
) -> tuple[np.ndarray, int, float]:
    """Stream raw PCM from POST /v1/audio/speech; return (audio, sr, ttfa_s)."""
    payload = {
        "input": text,
        "voice": speaker_id,
        "response_format": "pcm",
        "stream": True,
        "stream_format": "audio",
        "max_new_tokens": max_new_tokens,
    }
    t0 = time.perf_counter()
    t_first = None
    chunks: list[np.ndarray] = []
    trailing = b""
    with requests.post(SPEECH_ENDPOINT, json=payload, stream=True, timeout=timeout) as resp:
        resp.raise_for_status()
        for chunk in resp.iter_content(chunk_size=None):
            if not chunk:
                continue
            data = trailing + chunk
            even = len(data) - (len(data) % 2)
            trailing = data[even:]
            if even == 0:
                continue
            if t_first is None:
                t_first = time.perf_counter()
            pcm = np.frombuffer(data[:even], dtype="<i2").astype(np.float32) / 32768.0
            chunks.append(pcm)
    if not chunks:
        raise RuntimeError("empty audio response")
    audio = np.concatenate(chunks)
    ttfa = (t_first - t0) if t_first is not None else 0.0
    return audio, SAMPLE_RATE, ttfa

In [ ]:
TEXT = "Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows."

http_audio, sr, ttfa = synthesize_http(TEXT)
print(f"HTTP: {len(http_audio)} samples ({len(http_audio)/sr:.2f}s @ {sr} Hz), TTFA {ttfa*1000:.0f} ms")
display(Audio(http_audio, rate=sr))

## Incremental input with `/v1/audio/speech/stream`

The client tokenizes the full text with the converted EasyMagpie tokenizer, sends five token IDs per `input.tokens` event, and receives binary PCM frames concurrently. Exact IDs are preferable to independent `input.text` chunks because they preserve tokenizer boundaries.

In [ ]:
async def synthesize_incremental(
    text: str,
    speaker_id: str = "eng",
    tokens_per_chunk: int = 5,
    max_new_tokens: int = 1024,
) -> tuple[np.ndarray, int, float]:
    """Send token-ID chunks over WebSocket while receiving PCM asynchronously."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    token_chunks = [
        token_ids[index : index + tokens_per_chunk]
        for index in range(0, len(token_ids), tokens_per_chunk)
    ]

    pcm_parts: list[bytes] = []
    sample_rate = SAMPLE_RATE
    t0 = time.perf_counter()
    t_first = None

    async with websockets.connect(STREAM_ENDPOINT, max_size=64 * 1024 * 1024) as websocket:
        await websocket.send(
            json.dumps(
                {
                    "type": "session.config",
                    "voice": speaker_id,
                    "stream_audio": True,
                    "response_format": "pcm",
                    "max_new_tokens": max_new_tokens,
                }
            )
        )

        async def send_tokens() -> None:
            for chunk in token_chunks:
                await websocket.send(json.dumps({"type": "input.tokens", "tokens": chunk}))
            await websocket.send(json.dumps({"type": "input.done"}))

        sender = asyncio.create_task(send_tokens())
        try:
            while True:
                message = await websocket.recv()
                if isinstance(message, bytes):
                    if t_first is None:
                        t_first = time.perf_counter()
                    pcm_parts.append(message)
                    continue

                event = json.loads(message)
                event_type = event.get("type")
                if event_type == "audio.start":
                    sample_rate = int(event.get("sample_rate", sample_rate))
                elif event_type == "audio.done":
                    print(
                        "server metrics:",
                        f"talker_frames={event.get('talker_frames')}",
                        f"text_tokens={event.get('text_tokens')}",
                        f"audio_bytes={event.get('total_bytes')}",
                    )
                elif event_type == "error":
                    raise RuntimeError(event["message"])
                elif event_type == "session.done":
                    break
        finally:
            await sender

    if not pcm_parts:
        raise RuntimeError("empty audio response")
    pcm = b"".join(pcm_parts)
    audio = np.frombuffer(pcm, dtype="<i2").astype(np.float32) / 32768.0
    ttfa = (t_first - t0) if t_first is not None else 0.0
    return audio, sample_rate, ttfa

In [ ]:
incremental_audio, sr, ttfa = await synthesize_incremental(TEXT, tokens_per_chunk=5)
print(
    f"WebSocket: {len(incremental_audio)} samples "
    f"({len(incremental_audio)/sr:.2f}s @ {sr} Hz), TTFA {ttfa*1000:.0f} ms"
)
display(Audio(incremental_audio, rate=sr))